# Assignment 1: Perceptrons and Multi-Layer Perceptrons

**Deep Learning FS26**

---

## Part 1: Task Description

### Problem Description

In this assignment you will implement a Perceptron and a Multi-Layer Perceptron (MLP) from scratch using NumPy, and then explore the same models using PyTorch. The goal is to deepen your understanding of how neural networks learn through forward propagation and backpropagation.

### Tasks Overview

1. **Perceptron Implementation**
   - Implement a single Perceptron with a step activation function.
   - Train it on a linearly separable binary classification dataset (e.g., AND / OR gate).
   - Observe the decision boundary before and after training.

2. **Multi-Layer Perceptron (MLP)**
   - Implement a 2-layer MLP with sigmoid activations from scratch.
   - Train it using stochastic gradient descent (SGD) and backpropagation.
   - Solve the XOR problem, which a single perceptron cannot solve.

3. **PyTorch MLP**
   - Reimplement the MLP using `torch.nn.Module`.
   - Train using `torch.optim.SGD` and `nn.BCELoss`.
   - Compare loss curves with your manual implementation.

### Possible Solutions

- The Perceptron should correctly classify all AND/OR inputs after training (accuracy = 100%).
- The MLP should learn the XOR function — a task impossible for a single-layer network.
- The loss should decrease monotonically (or near-monotonically) across epochs.

### Expected Plots

Below are hints for the kinds of visualizations your solutions should produce:

- **Decision boundary plot** for the Perceptron: A linear boundary separating the two classes in 2D.
  - The boundary should clearly separate class 0 and class 1 after training.

- **Loss curve**: A plot of training loss vs. epoch showing a decreasing trend.
  - The y-axis is the loss (BCE or MSE), and the x-axis is the epoch number.

- **XOR decision boundary**: A non-linear boundary produced by the MLP separating the four XOR points.

```
Example: Expected loss curve shape

Loss
1.0 |*
    | *
0.5 |   **
    |     ****
0.0 |__________***__
     0    50   100  Epoch
```

## Part 2: Implementation

### Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# For reproducibility
np.random.seed(42)
torch.manual_seed(42)

### Task 1: Perceptron from Scratch

In [ ]:
# Dataset: AND gate
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])  # AND labels

class Perceptron:
    def __init__(self, n_features, lr=0.1):
        # TODO: Initialize weights and bias
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = lr

    def step(self, x):
        # TODO: Implement step activation function
        return 1 if x >= 0 else 0

    def predict(self, X):
        # TODO: Compute weighted sum + bias, apply activation
        linear_output = np.dot(X, self.weights) + self.bias
        return np.array([self.step(x) for x in linear_output])

    def fit(self, X, y, epochs=100):
        # TODO: Implement the perceptron learning rule
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                y_pred = self.step(np.dot(xi, self.weights) + self.bias)
                error = yi - y_pred
                self.weights += self.lr * error * xi
                self.bias += self.lr * error

# Train
perceptron = Perceptron(n_features=2)
perceptron.fit(X_and, y_and, epochs=100)
preds = perceptron.predict(X_and)
print("AND gate predictions:", preds)
print("Accuracy:", np.mean(preds == y_and))

In [ ]:
# Visualize decision boundary
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),
                         np.arange(y_min, y_max, 0.01))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='k', s=100)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.tight_layout()
    plt.show()

plot_decision_boundary(perceptron, X_and, y_and, "Perceptron Decision Boundary (AND gate)")

### Task 2: MLP from Scratch (XOR problem)

In [ ]:
# Dataset: XOR
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)

class MLP:
    def __init__(self, n_input, n_hidden, n_output, lr=0.5):
        # TODO: Initialize weights and biases (Xavier-style)
        self.W1 = np.random.randn(n_input, n_hidden) * 0.5
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(n_hidden, n_output) * 0.5
        self.b2 = np.zeros((1, n_output))
        self.lr = lr

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def sigmoid_deriv(self, a):
        return a * (1 - a)

    def forward(self, X):
        # TODO: Implement forward pass
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.sigmoid(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, X, y):
        # TODO: Implement backpropagation
        m = X.shape[0]
        dL_da2 = self.a2 - y
        da2_dz2 = self.sigmoid_deriv(self.a2)
        delta2 = dL_da2 * da2_dz2

        dW2 = self.a1.T @ delta2 / m
        db2 = delta2.mean(axis=0, keepdims=True)

        delta1 = (delta2 @ self.W2.T) * self.sigmoid_deriv(self.a1)
        dW1 = X.T @ delta1 / m
        db1 = delta1.mean(axis=0, keepdims=True)

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def fit(self, X, y, epochs=5000):
        losses = []
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = np.mean((y_pred - y) ** 2)
            losses.append(loss)
            self.backward(X, y)
        return losses

    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int)

mlp = MLP(n_input=2, n_hidden=4, n_output=1, lr=1.0)
losses = mlp.fit(X_xor, y_xor, epochs=5000)

preds = mlp.predict(X_xor)
print("XOR predictions:", preds.flatten())
print("Expected:       ", y_xor.flatten().astype(int))
print("Accuracy:", np.mean(preds == y_xor))

In [ ]:
# Plot training loss curve
plt.figure(figsize=(8, 4))
plt.plot(losses, color='steelblue', linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss — MLP on XOR (NumPy)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Task 3: PyTorch MLP

In [ ]:
# Prepare data as PyTorch tensors
X_t = torch.FloatTensor(X_xor)
y_t = torch.FloatTensor(y_xor)

class TorchMLP(nn.Module):
    def __init__(self, n_input, n_hidden, n_output):
        super(TorchMLP, self).__init__()
        # TODO: Define layers
        self.layer1 = nn.Linear(n_input, n_hidden)
        self.layer2 = nn.Linear(n_hidden, n_output)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # TODO: Implement forward pass
        x = self.sigmoid(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

model = TorchMLP(n_input=2, n_hidden=4, n_output=1)
optimizer = optim.SGD(model.parameters(), lr=1.0)
criterion = nn.BCELoss()

torch_losses = []
for epoch in range(5000):
    optimizer.zero_grad()
    y_pred = model(X_t)
    loss = criterion(y_pred, y_t)
    loss.backward()
    optimizer.step()
    torch_losses.append(loss.item())

with torch.no_grad():
    preds_torch = (model(X_t) >= 0.5).float()
print("PyTorch XOR predictions:", preds_torch.flatten().int().numpy())
print("Accuracy:", (preds_torch == y_t).float().mean().item())

In [ ]:
# Compare loss curves: NumPy vs PyTorch
plt.figure(figsize=(10, 4))
plt.plot(losses, label='NumPy MLP (MSE)', color='steelblue', linewidth=1.5)
plt.plot(torch_losses, label='PyTorch MLP (BCE)', color='coral', linewidth=1.5, linestyle='--')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Comparison — NumPy vs PyTorch")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Experiments and Analysis

In this section you will run experiments to better understand the behavior of perceptrons and MLPs.

### Experiment 1: Effect of Learning Rate

Train your NumPy MLP on the XOR problem with different learning rates (`0.01`, `0.1`, `1.0`, `5.0`) and compare the resulting loss curves.

In [ ]:
learning_rates = [0.01, 0.1, 1.0, 5.0]
plt.figure(figsize=(10, 5))

for lr in learning_rates:
    # TODO: Train MLP with each learning rate and plot the loss
    np.random.seed(42)
    model_lr = MLP(n_input=2, n_hidden=4, n_output=1, lr=lr)
    lr_losses = model_lr.fit(X_xor, y_xor, epochs=5000)
    plt.plot(lr_losses, label=f'lr={lr}', linewidth=1.5)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Effect of Learning Rate on Training Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO: Describe what you observe below
# - Which learning rate converges fastest?
# - Which causes instability?
# YOUR ANSWER HERE:

### Experiment 2: Effect of Hidden Layer Size

Train your MLP with different numbers of hidden units (`2`, `4`, `8`, `16`) and compare accuracy and loss.

In [ ]:
hidden_sizes = [2, 4, 8, 16]
plt.figure(figsize=(10, 5))

for h in hidden_sizes:
    # TODO: Train MLP with each hidden size and plot the loss
    np.random.seed(42)
    model_h = MLP(n_input=2, n_hidden=h, n_output=1, lr=1.0)
    h_losses = model_h.fit(X_xor, y_xor, epochs=5000)
    final_acc = np.mean(model_h.predict(X_xor) == y_xor)
    plt.plot(h_losses, label=f'hidden={h}, acc={final_acc:.2f}', linewidth=1.5)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Effect of Hidden Layer Size on Training Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO: Describe what you observe below
# YOUR ANSWER HERE:

### Experiment 3: XOR Decision Boundary Visualization

Visualize the decision boundary of your trained MLP on the XOR problem.

In [ ]:
# TODO: Plot the MLP decision boundary for XOR
np.random.seed(42)
mlp_final = MLP(n_input=2, n_hidden=4, n_output=1, lr=1.0)
mlp_final.fit(X_xor, y_xor, epochs=5000)

x_min, x_max = -0.5, 1.5
y_min, y_max = -0.5, 1.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),
                     np.arange(y_min, y_max, 0.01))
Z = mlp_final.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor.flatten(),
            cmap=plt.cm.RdYlBu, edgecolors='k', s=200, zorder=3)
for i, (xi, yi) in enumerate(zip(X_xor, y_xor)):
    plt.annotate(f'XOR={int(yi[0])}', xy=xi, xytext=(xi[0]+0.05, xi[1]+0.05))
plt.title("MLP Decision Boundary (XOR Problem)")
plt.xlabel("Input 1")
plt.ylabel("Input 2")
plt.tight_layout()
plt.show()

### Summary Questions

Answer the following questions in the cell below:

1. Why can a single Perceptron not solve the XOR problem?
2. What role does the hidden layer play in solving non-linearly separable problems?
3. How does the learning rate affect convergence speed and stability?
4. What happens if you initialize all weights to zero? Why?

**Your Answers:**

1. *TODO: Your answer here*

2. *TODO: Your answer here*

3. *TODO: Your answer here*

4. *TODO: Your answer here*